In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Google Sheets linkini her zaman GÜNCEL çekecek taktik:
# Linkin sonuna eklediğimiz '&v=' kısmı Google'ın size eski veriyi vermesini engeller.
import pandas as pd
import time

# Buraya kendi gid (sayfa id) numaranı ve linkini koy
timestamp = int(time.time()) # Her seferinde farklı bir sayı üretir
url = f"https://docs.google.com/spreadsheets/d/19rN6riciEulRHiPAU1Nr-1LHSn-xVQ_BaQ2orFDh21w/export?format=csv&gid=1887374425&v={timestamp}"

df = pd.read_csv(url)
print(f"✅ En güncel veri çekildi! Satır sayısı: {len(df)}")
# O 'kulp' yorumunu kontrol edelim:
print(df[df['review_text'].str.contains("kulp", na=False)])

# Kaç satır var
print(df.shape[0])

# Son 5 satır
print(df.tail(5))

# 'Sutun_Adi' yazan yere kendi sütun ismini yazmalısın
# 'Sutun_Adi' yazan yere kendi sütun ismini yazmalısın
sonuc = df[df['review_text'].str.contains('Ürün kesinlikle orijinal değil almayın ', case=False, na=False)]
print(sonuc)

df['Etiket'].value_counts()

✅ En güncel veri çekildi! Satır sayısı: 2651
                                            review_text   Etiket  \
0     tava iyide bu kulplar nedir böyle. tutmaya yür...  Alakalı   
10    Bu kadar övülen bu tavaların kulplarının isınd...  Alakalı   
73    Ürün güzel fakat kulpları da silikon olsaydı d...  Alakalı   
88    Kulpları çok ısındığı için mutlaka bir şey kul...  Alakalı   
90    Tavanın kendisi çok güzel fakat kulpları efsan...  Alakalı   
...                                                 ...      ...   
1862      Çok ince kulpu çabuk ısınıyor hiç güzel değil  Alakalı   
1874  Çok güzel fakat tavanın kulpları çok ısınıyor ...  Alakalı   
1899  Tavalar güzel yapıştırma yapmıyor ama kulpları...  Alakalı   
1967  Tava hiç yapışmıyor çok güzel fakat kulpları ç...  Alakalı   
1987  Hafif Bi ürün. Günlük kullanıma uygun fakatttt...  Alakalı   

      review_rating  
0                 3  
10                2  
73                3  
88                3  
90                3  
...   

,count
Etiket,
Alakalı,1454
Alakasız,1197


In [ ]:
# temizlik
import re

print("🧹 BERTurk İçin Minimalist Veri Temizliği Başlıyor...")

def bert_hafif_temizlik(metin):
    # Eğer metin boşsa (NaN), boş string döndür
    if not isinstance(metin, str):
        return ""

    # 1. Emojileri ve tuhaf sembolleri sil (Sadece harf, rakam ve !?., kalsın)
    metin = re.sub(r'[^\w\s\.,!?ığüşöçİĞÜŞÖÇ]', ' ', metin)

    # 2. Yan yana girilmiş 5-10 tane boşluğu, tek boşluğa indir
    metin = re.sub(r'\s+', ' ', metin).strip()

    return metin

# Fonksiyonu review_text sütununa uygula
df['review_text'] = df['review_text'].apply(bert_hafif_temizlik)

# Boş kalan satırları uçur
df = df[df['review_text'] != ""]

print("✅ Temizlik bitti! İlk 5 satırın yeni hali:")
print(df.head(5))

🧹 BERTurk İçin Minimalist Veri Temizliği Başlıyor...
✅ Temizlik bitti! İlk 5 satırın yeni hali:
                                         review_text   Etiket  review_rating
0  tava iyide bu kulplar nedir böyle. tutmaya yür...  Alakalı              3
1  Ürünü iki üç defa kullandım hafif bir ürün çiz...  Alakalı              3
2  Güzel yumuşak bir yastık, başınızı koyduğunuzd...  Alakalı              4
3  Beklentim belki de çok yüksek di ama hayal kır...  Alakalı              3
4  çok kaliteli bişey beklemeyin hafif hiç sarmad...  Alakalı              3


In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("🚀 ELECTRA PIPELINE BAŞLIYOR: (Türkçe Sürüm)\n")

# --- 1. VERİ HAZIRLIĞI ---
# Etiket sütununun 'Etiket' (büyük E) olduğunu varsayıyorum, hata alırsan küçük yaparsın.
df['Etiket_Sayisal'] = df['Etiket'].astype(str).str.strip().map({
    'Alakasız': 1, 'Alakalı': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0
})
df = df.dropna(subset=['review_text', 'Etiket_Sayisal'])

X = df['review_text'].tolist()
y = df['Etiket_Sayisal'].astype(int).tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_dataset = Dataset.from_dict({"text": X_train, "label": y_train})
test_dataset = Dataset.from_dict({"text": X_test, "label": y_test})

# --- 2. ELECTRA TOKENIZER ---
model_name = "dbmdz/electra-base-turkish-cased-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("⚙️ Veriler ELECTRA'nın anlayacağı dile (Token) çevriliyor...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=500)
test_dataset = test_dataset.map(tokenize_function, batched=True, batch_size=500)

# --- 3. MODELİ YÜKLEME ---
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# --- 4. METRİKLER ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# --- 5. EĞİTİM AYARLARI ---
training_args = TrainingArguments(
    output_dir="./electra_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# --- 6. ATEŞLE! ---
print("\n🔥 ELECTRA GPU Üzerinde Eğitiliyor...")
trainer.train()

print("\n🏆 ELECTRA TEST SETİ KARNESİ:")
sonuclar = trainer.evaluate()
for key, value in sonuclar.items():
    if key.startswith("eval_"):
        print(f"{key.replace('eval_', '').upper():<15} : %{value * 100:.2f}")

# --- 7. KAYDET ---
model_save_path = "./electra_production_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"\n✅ ELECTRA başarıyla '{model_save_path}' klasörüne kaydedildi.")

🚀 ELECTRA PIPELINE BAŞLIYOR: (Türkçe Sürüm)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye


🏆 ELECTRA TEST SETİ KARNESİ:


LOSS            : %13.99
ACCURACY        : %95.67
F1              : %95.20
PRECISION       : %95.40
RECALL          : %95.00
RUNTIME         : %346.39
SAMPLES_PER_SECOND : %15329.70
STEPS_PER_SECOND : %490.80


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ ELECTRA başarıyla './electra_production_model' klasörüne kaydedildi.


In [ ]:
# drive'a modeli ekledik
import shutil
import os

# Kaynak klasör (Colab'daki geçici yer)
kaynak = './electra_production_model'
# Hedef klasör (Drive'ındaki kalıcı yer)
hedef = '/content/drive/MyDrive/electra_production_model'

if os.path.exists(kaynak):
    # dirs_exist_ok=True parametresi: "Klasör varsa silme, içindekileri yenisiyle güncelle" demek.
    shutil.copytree(kaynak, hedef, dirs_exist_ok=True)
    print("✅ Şampiyon ConvBERT başarıyla Drive'a GÜNCELLENDİ ve taşındı!")
else:
    print("❌ Model klasörü bulunamadı, ismini kontrol et babuş!")

✅ Şampiyon ConvBERT başarıyla Drive'a GÜNCELLENDİ ve taşındı!
